<a href="https://colab.research.google.com/github/soham-never-codes/Festiva-AI-Event-Planner/blob/main/notebooks/Festiva_Week2_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#  Mount Drive (always first)
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/festiva'

os.makedirs(f'{PROJECT_DIR}/data/raw', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data/processed', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/rag', exist_ok=True)

print("✅ Drive mounted. Project folder architecture ready.")

Mounted at /content/drive
✅ Drive mounted. Project folder architecture ready.


In [2]:
#  Mount + imports

from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/festiva'

!pip install -q scikit-learn joblib

import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import joblib, warnings
warnings.filterwarnings('ignore')

print("✅ Environment ready!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Environment ready!


In [3]:
#  Load data + feature engineering

df = pd.read_csv(f'{PROJECT_DIR}/data/raw/event_budget_data.csv')

CAT_COLS = ['event_type', 'city', 'season']
FEATURE_COLS = ['event_type', 'city', 'guest_count', 'total_budget', 'duration_days', 'season']
SPEND_COLS = [c for c in df.columns if c not in FEATURE_COLS]

# Encode categoricals safely
encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

# Build target: percentage allocation per category
for col in SPEND_COLS:
    df[f'{col}_pct'] = df[col] / df['total_budget']

X = df[FEATURE_COLS].values
y = df[[f'{c}_pct' for c in SPEND_COLS]].fillna(0).values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print(f"✅ Train: {X_train.shape}, Test: {X_test.shape}, Targets: {y.shape[1]}")

✅ Train: (1600, 6), Test: (400, 6), Targets: 12


In [4]:
#  Train model

print("⚙️ Training Multi-Output Gradient Boosting Regressor...")

base = GradientBoostingRegressor(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)
model = MultiOutputRegressor(base, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_clipped = np.clip(y_pred, 0, None) # Prevent negative percentages

# Calculate safe metrics
mae = mean_absolute_error(y_test, y_pred_clipped)
r2 = r2_score(y_test, y_pred_clipped)

print(f"✅ Model Trained!")
print(f"📊 Mean Absolute Error: {mae:.4f} (Average error of ~{mae*100:.1f}% per category)")
print(f"📈 R² Score: {r2:.4f} (Higher is better, 1.0 is perfect)")

⚙️ Training Multi-Output Gradient Boosting Regressor...
✅ Model Trained!
📊 Mean Absolute Error: 0.0087 (Average error of ~0.9% per category)
📈 R² Score: 0.7985 (Higher is better, 1.0 is perfect)


In [5]:
#  Saving everything to Drive
import os
model_dir = f'{PROJECT_DIR}/models'
os.makedirs(model_dir, exist_ok=True)

joblib.dump(model,    f'{model_dir}/budget_model.pkl')
joblib.dump(scaler,   f'{model_dir}/scaler.pkl')
joblib.dump(encoders, f'{model_dir}/encoders.pkl')
joblib.dump({'feature_cols': FEATURE_COLS, 'spend_cols': SPEND_COLS},
            f'{model_dir}/metadata.pkl')

print(f"💾 All models and preprocessors saved to {model_dir}!")

💾 All models and preprocessors saved to /content/drive/MyDrive/festiva/models!


In [6]:
# Prediction function — test it

def predict_budget(event_type, city, total_budget, guest_count, duration_days=1, season='winter'):
    # Transform raw text inputs using our saved encoders
    row = np.array([[
        encoders['event_type'].transform([event_type])[0],
        encoders['city'].transform([city])[0],
        guest_count, total_budget, duration_days,
        encoders['season'].transform([season])[0]
    ]])

    # Scale and predict
    pcts = model.predict(scaler.transform(row))[0]
    pcts = np.clip(pcts, 0, None)

    # Normalize so all percentages exactly sum to 1.0 (100%)
    pcts /= pcts.sum()

    result = {}
    for cat, pct in zip(SPEND_COLS, pcts):
        # Only show categories that actually get a budget allocation > 0.5%
        if pct > 0.005:
            result[cat] = {'pct': round(pct*100, 1), 'amount': int(pct*total_budget)}

    # Sort by highest budget allocation
    return dict(sorted(result.items(), key=lambda x: -x[1]['amount']))

# --- Let's test it! ---
test_budget = 1_000_000
result = predict_budget('wedding', 'Bangalore', test_budget, 200)

print(f"\n🎉 Wedding in Bangalore — ₹{test_budget:,} budget breakdown:")
print("-" * 55)
for cat, v in result.items():
    bar = '█' * int(v['pct'] / 2)
    print(f"  {cat.capitalize():<15} | ₹{v['amount']:>8,}  ({v['pct']:>4.1f}%)  {bar}")


🎉 Wedding in Bangalore — ₹1,000,000 budget breakdown:
-------------------------------------------------------
  Venue           | ₹ 303,866  (30.4%)  ███████████████
  Catering        | ₹ 253,262  (25.3%)  ████████████
  Decoration      | ₹ 142,949  (14.3%)  ███████
  Photography     | ₹ 103,969  (10.4%)  █████
  Entertainment   | ₹  81,049  ( 8.1%)  ████
  Miscellaneous   | ₹  77,602  ( 7.8%)  ███
  Invitations     | ₹  36,685  ( 3.7%)  █
